# Agriculture — crop water demand and soil moisture

Agroclimatic studies typically pair **inputs** (precipitation,
irrigation potential), **demand** (reference evapotranspiration
$ET_0$), and **soil status** (root-zone moisture). ERA5-Land carries
all three at 0.1° resolution — finer than the standard ERA5 product
and tuned for land-surface processes.

**Domain context.** A simplified agronomic water balance for a
rain-fed field over one growing season:

$$ \text{deficit}(t) = \sum (P - |ET|)_t $$

When the cumulative deficit is negative, the crop is drawing on
stored soil moisture; when positive, soil moisture should be
recharging. Pairing the deficit time series with `soil_moisture_layer_1`
(top 7 cm) gives an immediate visual check on the model's coupling.

## Step 1 — confirm the variables and their flux flags

Soil moisture is a **state** field (volumetric water content,
instantaneous). Precipitation and evaporation are **flux** fields
(per-step accumulations). `op="auto"` routes each correctly without
the user picking.

In [ ]:
from earthly.ecmwf import Catalog

cat = Catalog()
for code in (
    "total-precipitation",
    "evaporation",
    "2m-temperature",
    "volumetric-soil-water-layer-1",
):
    spec = cat.get_variable("reanalysis-era5-land", code)
    print(
        f"{code:35s}  nc={spec.nc_variable:6s}  units={spec.units:25s}  is_flux={spec.is_flux}"
    )

## Step 2 — retrieve a growing season at monthly resolution

Box: a ~1° area in the central US corn belt (40°–41°N, 92°–91°W).
Range: April–October 2022 — sowing through harvest. ERA5-Land's
monthly-means product gives one value per month at 0.1° native grid.
Mixed flux + state retrieve in one call.

In [ ]:
from pathlib import Path
from earthly import Earthly, AggregationConfig

OUT = Path("data/era5-land-corn-belt")
OUT.mkdir(parents=True, exist_ok=True)

earthly = Earthly(
    data_source="ecmwf",
    temporal_resolution="monthly",
    start="2022-04-01",
    end="2022-10-01",
    variables={
        "reanalysis-era5-land-monthly-means": [
            "total-precipitation",
            "evaporation",
            "2m-temperature",
            "volumetric-soil-water-layer-1",
        ],
    },
    lat_lim=[40.0, 41.0],
    lon_lim=[-92.0, -91.0],
    cell_size=0.1,
    path=str(OUT),
)
earthly.download(aggregate=AggregationConfig(freq="1MS", op="auto", cell_size=0.1))

## Step 3 — assemble the deficit time series

Per-month catchment-mean for each variable, then derive the cumulative
$P - |ET|$ deficit and pair it with soil-moisture L1.

In [ ]:
import numpy as np
import pandas as pd
from pyramids.dataset import Dataset

agg = OUT / "aggregated"

def stack(cds_variable: str) -> np.ndarray:
    paths = sorted(agg.glob(f"{cds_variable}_1MS_*.tif"))
    return np.stack([Dataset.read_file(str(p)).read_array() for p in paths])

precip        = np.nanmean(stack("total_precipitation"),           axis=(1, 2)) * 1000  # mm
et            = -np.nanmean(stack("evaporation"),                  axis=(1, 2)) * 1000  # mm (sign flip)
t_air         = np.nanmean(stack("2m_temperature"),                axis=(1, 2)) - 273.15  # °C
soil_moisture = np.nanmean(stack("volumetric_soil_water_layer_1"), axis=(1, 2))         # m^3/m^3

months = pd.date_range("2022-04-01", periods=len(precip), freq="MS")
df = pd.DataFrame(
    {"P [mm]": precip.round(1), "|ET| [mm]": et.round(1), "P-|ET| [mm]": (precip - et).round(1),
     "deficit cum [mm]": (precip - et).cumsum().round(1),
     "T_2m [°C]": t_air.round(1), "SM_L1": soil_moisture.round(3)},
    index=months,
)
df

## Step 4 — soil moisture vs cumulative water surplus

If the model's land surface scheme is internally consistent, soil
moisture should track the cumulative $P - |ET|$ trend (with the soil's
natural drainage timescale).

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(months, precip - et, width=20, color="tab:green", alpha=0.5, label="P - |ET| [mm/month]")
ax1.set_ylabel("P - |ET| per month [mm]")
ax1.axhline(0, color="gray", lw=0.5)

ax2 = ax1.twinx()
ax2.plot(months, soil_moisture, marker="o", color="tab:brown", label="Soil moisture L1 [m³/m³]")
ax2.set_ylabel("Soil moisture L1 [m³/m³]", color="tab:brown")

ax1.set_title("Corn-belt growing season 2022 — water surplus and topsoil moisture")
fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.show()

## Notes

- **`reanalysis-era5-land`** is the ERA5 land-surface reanalysis at
  0.1° resolution — the right product for agronomic studies. ERA5
  single-levels lives at 0.25° and runs the same land scheme but with
  coarser surface fields.
- **Reference ET ($ET_0$).** ERA5 reports actual evaporation
  (model-computed). For Penman–Monteith $ET_0$, retrieve
  `surface-net-solar-radiation`, `2m-dewpoint-temperature`,
  `10m-wind-speed`, `surface-pressure` and run FAO-56 yourself; CDS
  doesn't expose $ET_0$ directly.
- **Volumetric soil moisture layers.** Layer 1 is 0–7 cm; layers 2/3/4
  are 7–28, 28–100, 100–289 cm. Pull all four for a root-zone profile.
- **Cell size.** This notebook passes `cell_size=0.1` (ERA5-Land
  native) on both `Earthly` and `AggregationConfig` so the GeoTIFFs
  carry the right pixel-size tag in their metadata.